In [2]:
import matplotlib.pyplot as plt
import pandas as pd 
import numpy as np
import string
import nltk
import nltk.corpus
import sklearn

from matplotlib import rcParams
from nltk.stem import WordNetLemmatizer 
from nltk.corpus import stopwords 
from nltk import NaiveBayesClassifier
from nltk.corpus import wordnet 
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk import pos_tag
from wordcloud import WordCloud
from sklearn.ensemble import RandomForestClassifier 
from nltk.classify.scikitlearn import SklearnClassifier

In [3]:
data_path = 'Womens Clothing E-Commerce Reviews.csv'
df=pd.read_csv(data_path)
df.head()

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [4]:
 
df.columns

Index(['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating',
       'Recommended IND', 'Positive Feedback Count', 'Division Name',
       'Department Name', 'Class Name'],
      dtype='object')

In [5]:
df.shape

(23486, 11)

In [7]:
# Count the number of unique values in each column
df.nunique()

Unnamed: 0                 23486
Clothing ID                 1206
Age                           77
Title                      13993
Review Text                22634
Rating                         5
Recommended IND                2
Positive Feedback Count       82
Division Name                  3
Department Name                6
Class Name                    20
dtype: int64

In [8]:
# Count the number of nulls in each column
df.isna().sum()

Unnamed: 0                    0
Clothing ID                   0
Age                           0
Title                      3810
Review Text                 845
Rating                        0
Recommended IND               0
Positive Feedback Count       0
Division Name                14
Department Name              14
Class Name                   14
dtype: int64

In [9]:
# drop all non values
df.dropna(inplace=True)

In [10]:
# reseat the index after droping some rows 
df.reset_index(drop=True, inplace=True)
df.head()

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
1,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
2,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses
3,5,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",2,0,4,General,Dresses,Dresses
4,6,858,39,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,5,1,1,General Petite,Tops,Knits


In [11]:
# Check for the missing values after droping the null values 
df.isnull().sum()

Unnamed: 0                 0
Clothing ID                0
Age                        0
Title                      0
Review Text                0
Rating                     0
Recommended IND            0
Positive Feedback Count    0
Division Name              0
Department Name            0
Class Name                 0
dtype: int64

In [12]:
# drop unnecessary culomns 
df.drop(["Unnamed: 0", "Title", 'Clothing ID'], axis=1, inplace=True)
df.head()

,Age,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,60,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
1,50,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
2,47,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses
3,49,"I love tracy reese dresses, but this one is no...",2,0,4,General,Dresses,Dresses
4,39,I aded this in my basket at hte last mintue to...,5,1,1,General Petite,Tops,Knits


In [13]:
# to remove spaces in columns and replace them with underscore 
df.columns= df.columns.str.replace(" ", "_")

In [14]:
 reviews = []
for (index , category) in enumerate(df.Recommended_IND):
    reviews.append((df.Review_Text[index],category)) # Store the review for spacific index with catogory inside texts array
reviews[0:4]

[('I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly into the zipper - it c',
  0),
 ("I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great compliments!",
  1),
 ('This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings and it is sleeveless so it pairs well with any cardigan. love this shirt!!!',
  1),
 ('I love tracy reese dresses, but this one is not for the very petite. i am just under 5 feet tall and usually wear a 0p in this brand. this dress 

We need to clean the reviews from stopwords `

In [15]:
# create lemmatizer 
lemmatizer = WordNetLemmatizer()
# Create a list of stopwords 
stops= stopwords.words("english")
punctuations=list(string.punctuation)
stops=stops+punctuations
stops, string.punctuation

(['i',
  'me',
  'my',
  'myself',
  'we',
  'our',
  'ours',
  'ourselves',
  'you',
  "you're",
  "you've",
  "you'll",
  "you'd",
  'your',
  'yours',
  'yourself',
  'yourselves',
  'he',
  'him',
  'his',
  'himself',
  'she',
  "she's",
  'her',
  'hers',
  'herself',
  'it',
  "it's",
  'its',
  'itself',
  'they',
  'them',
  'their',
  'theirs',
  'themselves',
  'what',
  'which',
  'who',
  'whom',
  'this',
  'that',
  "that'll",
  'these',
  'those',
  'am',
  'is',
  'are',
  'was',
  'were',
  'be',
  'been',
  'being',
  'have',
  'has',
  'had',
  'having',
  'do',
  'does',
  'did',
  'doing',
  'a',
  'an',
  'the',
  'and',
  'but',
  'if',
  'or',
  'because',
  'as',
  'until',
  'while',
  'of',
  'at',
  'by',
  'for',
  'with',
  'about',
  'against',
  'between',
  'into',
  'through',
  'during',
  'before',
  'after',
  'above',
  'below',
  'to',
  'from',
  'up',
  'down',
  'in',
  'out',
  'on',
  'off',
  'over',
  'under',
  'again',
  'further',
  'th

In [16]:
# business stopwords
business_stopwords= ["i'm","would", "look", "ordered", "wear", "fit", "one", "fits","bought", "looks", "also", "got", "think", "even",
                     "tried", "get", "could", "made","way","still", "runs","true" ,"right", "see","online","wearing", "however", "design","purchased","feel","go",
                     "enough","model","though","price","looked","person","better","first","going","try", "body" "bottom","time","many","looking","around","thought",
                     "make","wanted","saw","makes","went","find","found","buy","nan","i've", "since","seems","ok", "girl", "woman"]
stops= stops+business_stopwords
stops

['i',
 'me',
 'my',
 'myself',
 'we',
 'our',
 'ours',
 'ourselves',
 'you',
 "you're",
 "you've",
 "you'll",
 "you'd",
 'your',
 'yours',
 'yourself',
 'yourselves',
 'he',
 'him',
 'his',
 'himself',
 'she',
 "she's",
 'her',
 'hers',
 'herself',
 'it',
 "it's",
 'its',
 'itself',
 'they',
 'them',
 'their',
 'theirs',
 'themselves',
 'what',
 'which',
 'who',
 'whom',
 'this',
 'that',
 "that'll",
 'these',
 'those',
 'am',
 'is',
 'are',
 'was',
 'were',
 'be',
 'been',
 'being',
 'have',
 'has',
 'had',
 'having',
 'do',
 'does',
 'did',
 'doing',
 'a',
 'an',
 'the',
 'and',
 'but',
 'if',
 'or',
 'because',
 'as',
 'until',
 'while',
 'of',
 'at',
 'by',
 'for',
 'with',
 'about',
 'against',
 'between',
 'into',
 'through',
 'during',
 'before',
 'after',
 'above',
 'below',
 'to',
 'from',
 'up',
 'down',
 'in',
 'out',
 'on',
 'off',
 'over',
 'under',
 'again',
 'further',
 'then',
 'once',
 'here',
 'there',
 'when',
 'where',
 'why',
 'how',
 'all',
 'any',
 'both',
 'each

In [17]:
# function to get the simpler virsion of pos tag  to use it in lemmitazation 
def get_simple_pos(tag):
    if tag.startswith('N') or tag.startswith('J'):
        return wordnet.NOUN
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN #default case

In [19]:
# function to return Limmitzed words and cleaned from stop words  
def clean_review(words):
    output_words= []
    words_tokens= nltk.word_tokenize(words)  
    for word in words_tokens :

        if word.lower() not in stops:
            pos = pos_tag([word]) # get the part of speech of each word 

            clean_word=(lemmatizer.lemmatize(word.lower(), 
                                             pos=get_simple_pos(tag)) 
                        for word, tag in pos)
            output_words.append(', '.join(map(str,clean_word )))
    return output_words

In [26]:
#Test our function 
import nltk
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt')
nltk.download('omw-1.4')
nltk.download('wordnet')


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [25]:
clean_review("These dresses are so beautiful and durable")

['dress', 'beautiful', 'durable']

In [27]:
# clean all reviews 
cleaned_reviews= [(clean_review(text),category )for text,category  in reviews]

In [28]:
len(cleaned_reviews)

19662

In [29]:
#check first 2 reviews 
cleaned_reviews[0:2]

[(['high',
   'hope',
   'dress',
   'really',
   'work',
   'initially',
   'petite',
   'small',
   'usual',
   'size',
   'outrageously',
   'small',
   'small',
   'fact',
   'zip',
   'reorder',
   'petite',
   'medium',
   'ok.',
   'overall',
   'top',
   'half',
   'comfortable',
   'nicely',
   'bottom',
   'half',
   'tight',
   'layer',
   'several',
   'somewhat',
   'cheap',
   'net',
   'layer',
   'imo',
   'major',
   'flaw',
   'net',
   'layer',
   'sewn',
   'directly',
   'zipper',
   'c'],
  0),
 (['love',
   'love',
   'love',
   'jumpsuit',
   "'s",
   'fun',
   'flirty',
   'fabulous',
   'every',
   'nothing',
   'great',
   'compliment'],
  1)]

In [30]:
#.75% traning = 14746 and 25% testing = 19662-14746 =4916 
traning_words=cleaned_reviews[0:14746]
testing_words=cleaned_reviews[14746:]

In [31]:
# array contaning all words 
words_list=[]
for word in traning_words:
        words_list+=word[0] # 0 index to get only the words 

In [32]:
# Total words in traning data 
len(words_list)

392729

In [33]:
#frequency distribution for all words 
freq= nltk.FreqDist(words_list)
common=freq.most_common()
# features are an array of only the top words in word list without The number of words 
features= [i[0]for i in common]

In [34]:
print(len(common))
print(len(features))
# Most common 5 words 
common[0:5]

12846
12846


[('dress', 8091), ('size', 7083), ('love', 6701), ("n't", 5543), ('top', 5449)]

In [35]:
# List of 5 features 
features[0:5]

['dress', 'size', 'love', "n't", 'top']

In [42]:
# function to return a set of the features with true or false 
def get_dict_for_feature(words):
    current_features={}
    words_set= set(words)
    for word in features:
        current_features[word] = word in words_set          # if word comes in words set it will return True otherwise False 
    return current_features

In [43]:
featuers_dic= get_dict_for_feature(traning_words[0][0])
# Dictionary containing all words with True classification if the word is exist in each review otherwise false  
featuers_dic 

{'dress': True,
 'size': True,
 'love': False,
 "n't": False,
 'top': True,
 "'s": False,
 'like': False,
 'color': False,
 'great': False,
 "'m": False,
 '5': False,
 "''": False,
 'fabric': False,
 'small': True,
 'really': True,
 'perfect': False,
 'little': False,
 'flatter': False,
 'soft': False,
 'well': False,
 'back': False,
 'comfortable': True,
 'nice': False,
 'cute': False,
 'work': True,
 'bit': False,
 'sweater': False,
 'shirt': False,
 'beautiful': False,
 'material': False,
 'large': False,
 'much': False,
 'jean': False,
 'length': False,
 'petite': True,
 'long': False,
 'short': False,
 'waist': False,
 'medium': True,
 'quality': False,
 'retailer': False,
 'x': False,
 'pretty': False,
 'skirt': False,
 'pant': False,
 'store': False,
 'usually': False,
 'sleeve': False,
 '...': False,
 'style': False,
 'good': False,
 'cut': False,
 'big': False,
 'black': False,
 'return': False,
 'super': False,
 'picture': False,
 'say': False,
 'need': False,
 '4': False,
 '

In [44]:
# create dic for each review wich has feature with vlue and the category
traning_words= [( get_dict_for_feature(words),category ) for words , category in traning_words]
testing_words = [( get_dict_for_feature(words),category ) for words , category in testing_words]

# NaiveBayes classifier

In [45]:
# to the classifier we need to use NaiveBayesClassifier and pass the training words to it 
NB_classifier= NaiveBayesClassifier.train(traning_words)
print("classifier accuracy percent:",(nltk.classify.accuracy(NB_classifier, traning_words))*100)

classifier accuracy percent: 91.93001491930015


In [46]:
NB_classifier.show_most_informative_features(10)

Most Informative Features
               defective = True                0 : 1      =     26.3 : 1.0
            additionally = True                0 : 1      =     25.7 : 1.0
                 bizarre = True                0 : 1      =     22.7 : 1.0
                  poorly = True                0 : 1      =     21.8 : 1.0
                  nipple = True                0 : 1      =     16.6 : 1.0
                    blah = True                0 : 1      =     16.2 : 1.0
              unwearable = True                0 : 1      =     16.2 : 1.0
                   worst = True                0 : 1      =     16.1 : 1.0
                    asap = True                0 : 1      =     15.4 : 1.0
                   shame = True                0 : 1      =     15.3 : 1.0


In [50]:

def test_custom_review(reviews_list, classifier):
    
    for idx,review in enumerate(reviews_list) : 
        custom_tokens = clean_review(review)
        print(f"The clean review is : "  , str(custom_tokens).replace('[','').replace(']',''))
        classifiers=classifier.classify(dict([token, True] for token in custom_tokens))
        if (classifiers == 1):
            pred = "Positive"
        else:
            pred = "Negative"
        print(f"Review number {idx +1 }  seems to be {pred} \n")

In [55]:

review_1 = " I honestly thought I’d have to send this back. Thought I’d look way to huge or like a marshmallow. But it’s cute and looks good! Perfect for cold days. Although I did order a size down and that was perfect. "
review_2= "I love these jeans! They fit perfectly – with a bit of relax, but still very slimming and flattering. Was happy that they didn’t look “painted” on as I’m not exactly 16 anymore lol. And of course they are high quality, as I would expect from Levis. Very reasonably priced too!"
review_3="The shirt was advertised as a slim fit, but the sizing was all wrong. It was way too big and looked sloppy on me."
review_4="The quality of the shirt was really disappointing. I could buy a similar shirt for half the price at a different store."
review_5="The color of this shirt is even better in person! It's a beautiful shade of blue and makes me feel confident wearing it."
review_6="This sweater is pretty great. I love the length and the fit and mostly I love how cozy it is. It keeps me warm, looks great and is soft and luxurious. I am thinking about getting another in a different color…because if it’s this good, why not?"
reviews = [review_1,review_2,review_3,review_4,review_5, review_6]


test_custom_review(reviews,NB_classifier)

The clean review is :  'honestly', '’', 'send', 'back', '’', 'huge', 'like', 'marshmallow', '’', 'cute', 'good', 'perfect', 'cold', 'day', 'although', 'order', 'size', 'perfect'
Review number 1  seems to be Negative 

The clean review is :  'love', 'jean', 'perfectly', '–', 'bit', 'relax', 'slimming', 'flatter', 'happy', '’', '“', 'paint', '”', '’', 'exactly', '16', 'anymore', 'lol', 'course', 'high', 'quality', 'expect', 'levi', 'reasonably', 'price'
Review number 2  seems to be Positive 

The clean review is :  'shirt', 'advertised', 'slim', 'size', 'wrong', 'big', 'sloppy'
Review number 3  seems to be Negative 

The clean review is :  'quality', 'shirt', 'really', 'disappoint', 'similar', 'shirt', 'half', 'different', 'store'
Review number 4  seems to be Negative 

The clean review is :  'color', 'shirt', "'s", 'beautiful', 'shade', 'blue', 'confident'
Review number 5  seems to be Positive 

The clean review is :  'sweater', 'pretty', 'great', 'love', 'length', 'mostly', 'love', 'co